# Figure 3: Example aircraft contrail

The width information to reproduce figure 3 is already processed and stored. 

To get width information for other contrails, run `figure_7.ipynb`. The zoom-in images are already processed before and saved to `../data/figure_3_zoom/` (`figure_3_zoom_ac.png`, `figure_3_zoom_detector.png` and `figure_3_zoom_vortex.png`). 

**Note:** Running this code the first time might take longer since the satellite data needs to be downloaded from the Google Sentinel-2 dataset. 

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyproj
import warnings
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=RuntimeWarning)
from matplotlib.patches import Polygon as MplPolygon
from PIL import Image
from scipy import ndimage
from shapely.geometry import MultiPolygon, Polygon, shape
from shapely.ops import transform

from pycontrails.datalib import sentinel


## Helper functions

In [ ]:
def rotate_point(x, y, cx, cy, angle_deg):
    """
    Rotate a point (x, y) around center (cx, cy) by angle_deg degrees.
    Positive angles rotate counter-clockwise.
    """
    # Convert angle to radians
    theta = np.deg2rad(angle_deg)
    
    # Translate point to origin
    x_shifted = x - cx
    y_shifted = y - cy
    
    # Rotate
    x_rot = x_shifted * np.cos(theta) - y_shifted * np.sin(theta)
    y_rot = x_shifted * np.sin(theta) + y_shifted * np.cos(theta)
    
    # Translate back
    x_new = x_rot + cx
    y_new = y_rot + cy
    return x_new, y_new

# Function to convert UTM to pixel coordinates
def utm_to_pixel(x_utm, y_utm, x_min, y_max, resolution):
    px = (x_utm - x_min) / resolution
    py = (y_max - y_utm) / resolution  # invert y-axis (top-left origin)
    return px, py

def rotate_coords(coords, cx, cy, angle_deg):
    rotated = []
    for (x, y) in coords:
        x_r, y_r = rotate_point(x, y, cx, cy, angle_deg)
        rotated.append((x_r, y_r))
    return rotated

def rotate_polygon(poly, cx, cy, angle_deg):
    # Rotate exterior ring
    exterior = rotate_coords(poly.exterior.coords, cx, cy, angle_deg)

    # Rotate interior rings (holes), if present
    interiors = []
    for interior in poly.interiors:
        interiors.append(rotate_coords(interior.coords, cx, cy, angle_deg))

    return Polygon(exterior, interiors)

def rotate_geometry(geom, cx, cy, angle_deg):
    if geom.geom_type == "Polygon":
        return rotate_polygon(geom, cx, cy, angle_deg)

    elif geom.geom_type == "MultiPolygon":
        polys = [rotate_polygon(p, cx, cy, angle_deg) for p in geom.geoms]
        return MultiPolygon(polys)

    else:
        raise ValueError(f"Geometry type {geom.geom_type} not supported.")
    
def polygon_to_pixel(poly, x_min, y_max, resolution):
    pixel_coords = []
    for (x, y) in poly.exterior.coords:
        xp, yp = utm_to_pixel(x, y, x_min, y_max, resolution)
        pixel_coords.append((xp, yp))
    return pixel_coords

def rotate_pixel_polygon(pixel_poly, cx, cy, angle_deg):
    rotated = []
    for (x, y) in pixel_poly:
        xr, yr = rotate_point(x, y, cx, cy, angle_deg)
        rotated.append((xr, yr))
    return rotated

def polygon_to_pixel_coords(poly, x_min, y_max, res):
    """Convert a Shapely polygon in UTM to a list of pixel coordinates."""
    coords_pix = []
    for (x, y) in poly.exterior.coords:
        xp, yp = utm_to_pixel(x, y, x_min, y_max, res)
        coords_pix.append((xp, yp))
    return coords_pix

In [ ]:
def load_sentinel_mosaic(base_urls, granule_ids, band="B10"):
    # Load multiple Sentinel tiles and combine into one mosaic
    datasets = []

    for base_url, granule_id in zip(base_urls, granule_ids):
        handler = sentinel.Sentinel(base_url, granule_id, bands=[band])
        datasets.append(handler.get())

    mosaic = datasets[0]
    for ds in datasets[1:]:
        mosaic = ds.combine_first(mosaic)

    return mosaic

## Download and rotate cirrus-band (subfigures d & e)

In [ ]:
# Sentinel tile configuration
base_urls = [
    "gs://gcp-public-data-sentinel-2/tiles/25/T/DE/S2A_MSIL1C_20210911T131031_N0301_R038_T25TDE_20210911T150612.SAFE",
    "gs://gcp-public-data-sentinel-2/tiles/25/T/EE/S2A_MSIL1C_20210911T131031_N0301_R038_T25TEE_20210911T150612.SAFE",
    "gs://gcp-public-data-sentinel-2/tiles/25/T/FE/S2A_MSIL1C_20210911T131031_N0301_R038_T25TFE_20210911T150612.SAFE",
]

granule_ids = [
    "L1C_T25TDE_A032496_20210911T131028",
    "L1C_T25TEE_A032496_20210911T131028",
    "L1C_T25TFE_A032496_20210911T131028",
]

ROTATION_ANGLE = -18.5  # degrees
RESOLUTION = 60      # meters per pixel

df = pd.read_csv('../data/landsat_sentinel_collocations_20260216.csv', comment='#')

# Specify the granule_id
granule_id = 'L1C_T25SDD_A032496_20210911T131028'
row = df.loc[df['scene_id'] == granule_id]

x_label = row["x"].item()
y_label = row["y"].item()
contrail_label = row["contrail_label"].item()
contrail_label = json.loads(contrail_label)

In [ ]:
handler = sentinel.Sentinel(base_urls[0], granule_ids[0], bands=["B10"])

In [ ]:
mosaic = load_sentinel_mosaic(base_urls, granule_ids)

img = mosaic["B10"].values

# Normalize image to uint8
img_norm = (img - np.nanmin(img)) / (np.nanmax(img) - np.nanmin(img))
img_uint8 = (img_norm * 255).astype(np.uint8)

x_min, x_max = mosaic["x"].values.min(), mosaic["x"].values.max()
y_min, y_max = mosaic["y"].values.min(), mosaic["y"].values.max()

img_height, img_width = img_uint8.shape
cx, cy = img_width / 2, img_height / 2

In [ ]:
# convert from ds to np array figure
img = mosaic["B10"].values  # 2D array, shape (ny, nx)

img_norm = (img - np.nanmin(img)) / (np.nanmax(img) - np.nanmin(img)) # Normalize to 0–1
img = (img_norm * 255).astype(np.uint8) # Convert to uint8 0–255 if you want an actual image file

In [ ]:
x_min, x_max = mosaic["x"].values.min(), mosaic["x"].values.max()
y_min, y_max = mosaic["y"].values.min(), mosaic["y"].values.max()

# Compute image size in pixels
img_height, img_width = img.shape[0], img.shape[1]

# Convert manual aircraft label to pixels
x_pixel, y_pixel = utm_to_pixel(x_label, y_label, x_min, y_max, RESOLUTION)

# Original image center
cx, cy = img_width / 2, img_height / 2

# Rotate point
x_rotated, y_rotated = rotate_point(x_pixel, y_pixel, cx, cy, -ROTATION_ANGLE)

rotated_img = ndimage.rotate(img, ROTATION_ANGLE, reshape=False)

fig, ax_img = plt.subplots(1, 1)  # Image on its own axis

ax_img.imshow(rotated_img, origin="upper", aspect=2, vmin=50, vmax=200)
ax_img.scatter(x_rotated, y_rotated, color="red")
ax_img.set_xlim(x_rotated - 100, x_rotated + 3000)
ax_img.set_ylim(y_rotated + 100, y_rotated - 100)
ax_img.axis("off")  # hide axes for the image

In [ ]:
# Project from WGS84 to the x and y coordinates in the UTM coordinate system
transformer = pyproj.Transformer.from_crs("EPSG:4326", handler.get_crs(), always_xy=True)

# Convert GeoJSON to a Shapely geometry
geom = shape(contrail_label)

# Transform the geometry to the local CRS
geom_utm = transform(transformer.transform, geom)

if geom_utm.geom_type == "Polygon":
    pixel_polys = [polygon_to_pixel_coords(geom_utm, x_min, y_max, RESOLUTION)]

elif geom_utm.geom_type == "MultiPolygon":
    pixel_polys = [
        polygon_to_pixel_coords(p, x_min, y_max, RESOLUTION)
        for p in geom_utm.geoms
    ]

else:
    raise ValueError("Unsupported geometry type")

pixel_polys_rot = [
    rotate_pixel_polygon(pp, cx, cy, -ROTATION_ANGLE)
    for pp in pixel_polys
]

In [ ]:
fig, ax_img = plt.subplots(1, 1, figsize=(12, 4))  # Image on its own axis

ax_img.imshow(rotated_img, origin="upper", aspect=2, vmin=50, vmax=200)
ax_img.scatter(x_rotated, y_rotated, color="red")
ax_img.set_xlim(x_rotated - 100, x_rotated + 3000)
ax_img.set_ylim(y_rotated + 50, y_rotated - 50)
ax_img.axis("off")  # hide axes for the image

for coords in pixel_polys_rot:
    coords = np.array(coords)

    # filled polygon
    ax_img.add_patch(
        MplPolygon(
            coords,
            closed=True,
            facecolor="red",
            alpha=0.4,
            edgecolor="none",
            zorder=3
        )
    )

    # outline
    ax_img.add_patch(
        MplPolygon(
            coords,
            closed=True,
            fill=False,
            edgecolor="red",
            linewidth=2,
            alpha=0.9,
            zorder=4
        )
    )

## Download RGB images for zoom-ins

Downloading and handling the RGB image was a bit cumbersome, so I pre-processed this and saved the zoom-ins already in the `../data/figure_3_zoom` folder.

In [ ]:
img_aircraft = Image.open("../data/figure_3_zoom/figure_3_zoom_ac.png")
img_aircraft = np.array(img_aircraft).astype(np.float32) / 255.0

img_detector = Image.open("../data/figure_3_zoom/figure_3_zoom_detector.png")
img_detector = np.array(img_detector).astype(np.float32) / 255.0

img_vortex = Image.open("../data/figure_3_zoom/figure_3_zoom_vortex.png")
img_vortex = np.array(img_vortex).astype(np.float32) / 255.0

## Downloading the width data

Note: first run `figure_7.ipynb` to process the annotations to width evolution json files.

In [ ]:
duration = pd.Timedelta(seconds=600)

with open(f"../data/contrail_evolution/annotations/{granule_id}.json", 'r') as f:
    data = json.load(f)

width = pd.DataFrame({
    "age": pd.to_numeric(data["age"], errors="coerce"),
    "width": pd.to_numeric(data["width"], errors="coerce"),
})
width = width.dropna(subset=['width'])
width = width[width['age'] >= 1]
width = width[width['age'] <= duration.total_seconds()]
width = width.sort_values('age')

## Plotting the final figure

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import ConnectionPatch
from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset
import matplotlib.patches as mpatches
from matplotlib.patches import Polygon as MplPolygon
import numpy as np
import matplotlib.gridspec as gridspec

plt.rcParams.update({
    "axes.titlesize": 14,
    "axes.labelsize": 14,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 14
})

fig = plt.figure(figsize=(14, 6), dpi=300)

outer_gs = fig.add_gridspec(
    nrows=2, ncols=1,
    height_ratios=[2.0, 1.0],   # adjust as needed
    hspace=0.10                 # spacing *only* between top block and bottom
)

top_gs = gridspec.GridSpecFromSubplotSpec(
    nrows=2, ncols=1,
    subplot_spec=outer_gs[0],
    hspace=0.0                  # remove spacing between the top two images
)

ax_img = fig.add_subplot(top_gs[0])
ax_img_label = fig.add_subplot(top_gs[1])
ax_width = fig.add_subplot(outer_gs[1])

# Subplot d: Full contrail
im = ax_img.imshow(rotated_img, origin="upper", aspect=2) 
ax_img.set_xlim(x_rotated - 50, x_rotated + 1740)
ax_img.set_ylim(y_rotated + 30, y_rotated - 50)
ax_img.text(1.01, 0.95, "(d)", transform=ax_img.transAxes,
            fontsize=14, fontweight='bold', va='top')
ax_img.axis("off")

# Subplot a: Zoom-in on aircraft
ax_ins1 = inset_axes(
    ax_img,
    width=4.4,
    height=2.2,
    bbox_to_anchor=(0.01, 1.2, 1, 1),  # left=0, top=1 in parent coords
    bbox_transform=ax_img.transAxes,
    loc="lower left"
)
x1_min, x1_max = x_rotated - 20, x_rotated + 50
y1_min, y1_max = y_rotated - 20, y_rotated + 20
ax_ins1.set_xlim(x1_min, x1_max)
ax_ins1.set_ylim(y1_min, y1_max)
ax_ins1.imshow(img_aircraft, origin="upper", extent=[x1_min, x1_max, y1_min, y1_max])
ax_ins1.axis("off")
mark_inset(ax_img, ax_ins1, loc1=4, loc2=2, fc="none", ec="gray", linestyle='--')

# Subplot b: Zoom-in on detector border
ax_ins2 = inset_axes(
    ax_img,
    width=4.4,
    height=2.2,
    bbox_to_anchor=(0.343, 1.2, 1, 1),  # left=0, top=1 in parent coords
    bbox_transform=ax_img.transAxes,
    loc="lower left"
)
x1_min, x1_max = x_rotated + 565, x_rotated + 635
y1_min, y1_max = y_rotated - 30, y_rotated + 10
ax_ins2.set_xlim(x1_min, x1_max)
ax_ins2.set_ylim(y1_min, y1_max)
ax_ins2.imshow(img_detector, origin="upper", extent=[x1_min, x1_max, y1_min, y1_max])
ax_ins2.axis("off")
mark_inset(ax_img, ax_ins2, loc1=4, loc2=2, fc="none", ec="gray", linestyle='--')

# Subplot c: Zoom-in on vortex breakdown
ax_ins3 = inset_axes(
    ax_img,
    width=4.4,
    height=2.2,
    bbox_to_anchor=(0.65, 1.2, 1, 1),  # left=0, top=1 in parent coords
    bbox_transform=ax_img.transAxes,
    loc="lower left"
)
x1_min, x1_max = x_rotated + 1400, x_rotated + 1500
y1_min, y1_max = y_rotated - 58, y_rotated + 12
ax_ins3.set_xlim(x1_min, x1_max)
ax_ins3.set_ylim(y1_min, y1_max)
ax_ins3.imshow(img_vortex, origin="upper", extent=[x1_min, x1_max, y1_min, y1_max])
ax_ins3.axis("off")
mark_inset(ax_img, ax_ins3, loc1=3, loc2=4, fc="none", ec="gray", linestyle='--')

# Subplot e: Full label
ax_img_label.imshow(rotated_img, origin="upper", aspect=2)
ax_img_label.set_xlim(x_rotated - 50, x_rotated + 1740)
ax_img_label.set_ylim(y_rotated + 30, y_rotated - 50)
ax_img_label.axis("off")
ax_img_label.text(1.01, 0.95, "(e)", transform=ax_img_label.transAxes,
            fontsize=14, fontweight='bold', va='top')

for coords in pixel_polys_rot:
    coords = np.array(coords)
    ax_img_label.add_patch(
        MplPolygon(coords, closed=True, facecolor="red", alpha=0.7, edgecolor="none", zorder=3)
    )
    ax_img_label.add_patch(
        MplPolygon(coords, closed=True, fill=False, edgecolor="red", linewidth=2, alpha=0.7, zorder=4)
    )
legend_patch = mpatches.Patch(facecolor="red", edgecolor="red", alpha=0.7, label="Labelled Contrail")
# ax_img_label.legend(handles=[legend_patch], loc="lower right", fontsize=12)

# Subplot f: Contrail width
ax_width.plot(width["age"], width["width"], color="green", linewidth=2)
ax_width.set_ylabel("Contrail Width (m)")
ax_width.grid(alpha=0.3)
ax_width.text(1.01, 0.95, "(f)", transform=ax_width.transAxes, fontsize=14, fontweight='bold', va='top')
ax_width.set_ylim(0, 1000)
ax_width.set_xlim(-10, 400)
ax_width.set_xlabel("Contrail Age (s)")

# Colorbar 
cbar_ax = fig.add_axes([-0.01, 0.47, 0.015, 0.4])
cbar = fig.colorbar(im, cax=cbar_ax, orientation='vertical')
phys_min, phys_max = 0, 0.35
tick_values = np.linspace(phys_min, phys_max, 8)
locs = (tick_values - phys_min) / (phys_max - phys_min) * 255
cbar.set_ticks(locs)
cbar.set_ticklabels([f"{val:.2f}" for val in tick_values])
cbar.set_label("B10 Reflectance", rotation=90, labelpad=15)
cbar.ax.yaxis.set_label_position('left')

ax_ins1.text(0.92, 1.1, "(a)", transform=ax_ins1.transAxes, fontsize=14, fontweight='bold', va='top')
ax_ins2.text(0.92, 1.1, "(b)", transform=ax_ins2.transAxes, fontsize=14, fontweight='bold', va='top')
ax_ins3.text(0.92, 1.1, "(c)", transform=ax_ins3.transAxes, fontsize=14, fontweight='bold', va='top')

ax_ins2.text(0.40, 0.9, "Detector border", transform=ax_ins2.transAxes, fontsize=12, va='top', color="white")
ax_ins2.text(0.07, 0.45, "Contrail width", transform=ax_ins2.transAxes, fontsize=12, va='top', color="white")

ax_ins2.plot([x_rotated + 580, x_rotated + 580], [y_rotated - 8, y_rotated - 6], color="white", linewidth=1)
ax_ins2.plot([x_rotated + 578, x_rotated + 582], [y_rotated - 8, y_rotated - 8], color="white", linewidth=1)
ax_ins2.plot([x_rotated + 578, x_rotated + 582], [y_rotated - 6, y_rotated - 6], color="white", linewidth=1)

ax_ins2.annotate("", 
            xy=(x_rotated + 600, y_rotated - 4), 
            xytext=(x_rotated + 600, y_rotated + 2), 
            arrowprops=dict(arrowstyle='->', lw=1, color='white'))

ax_img_label.text(0.7, 0.3, "Annotated Contrail", transform=ax_img_label.transAxes, fontsize=14, va='top', color="red", fontweight='bold')
ax_img_label.annotate("", 
            xy=(x_rotated + 1080, y_rotated - 4), 
            xytext=(x_rotated + 1180, y_rotated + 8), 
            arrowprops=dict(arrowstyle='->', lw=3, color='red'))

ax_width.text(0.2, 0.83, "Detector \nBorders", transform=ax_width.transAxes, fontweight="bold", fontsize=14, va='top', color="darkorange")
ax_width.annotate("", 
            xy=(130, 600), 
            xytext=(100, 700), 
            arrowprops=dict(arrowstyle='->', lw=2, color='darkorange'))

for x in [568, 1045, 1520, 1996]:
    x = x + 66
    ax_img_label.plot([x + 650, x - 650], [0, 2000], color="darkorange", linestyle='--', linewidth=2, alpha=0.9)
    ax_img.plot([x + 650, x - 650], [0, 2000], color="darkorange", linestyle='--', linewidth=2, alpha=0.9)

for x in [30, 138, 248, 357]:
    ax_width.axvline(x=x, linestyle='--', linewidth=2, color="darkorange", alpha=0.9)

# final layout tweaks (GridSpec already gives good control)
plt.subplots_adjust(left=0.06, right=0.98, top=0.98, bottom=0.04)  # tweak margins
plt.savefig("../figures/fig03.png", dpi=300, bbox_inches='tight', pad_inches=0.1)

plt.show()